# Calcula tu inflación — por marca

Calcula la **inflación real por marca** con los datos abiertos de *Quién es Quién
en los Precios* (Profeco), y detecta **reduflación**: cuando el producto cuesta
casi lo mismo pero trae menos contenido.

**No necesitas saber Python ni instalar nada.** Ejecuta las celdas en orden con
el botón ▶ de la izquierda (o `Shift + Enter`).

### Antes de empezar: sube tus datos a Google Drive

Sube a tu Google Drive los archivos de QQP, **sin descomprimir**. Ponlos en una
carpeta llamada `QQP`:

```
Mi unidad/
└── QQP/
    ├── QQP_2024.rar
    └── QQP_2025.rar
```

Subir los `.rar` es mucho más rápido que subir los CSV: el `.rar` pesa unos
100 MB y adentro trae el año completo. Este cuaderno los descomprime por ti.

> Si prefieres, también puedes subir los CSV ya descomprimidos a esa carpeta.
> El cuaderno acepta las dos formas.

---

## Paso 1 — Preparar

Tarda medio minuto.

In [ ]:
#@title Paso 1: preparar (ejecuta esta celda)
!pip install -q pandas
!apt-get -qq install -y unar > /dev/null 2>&1
!wget -q -O inflacion_por_marca.py https://raw.githubusercontent.com/carsam68-MonheyB/CALCULA_TU_INFLACI-N/main/inflacion_por_marca.py

import os, shutil
os.makedirs("datos", exist_ok=True)
print("Listo." if shutil.which("unar") else
      "Listo, pero no se instalo el descompresor: tendras que subir los CSV ya descomprimidos.")

---

## Paso 2 — Conectar tu Google Drive

Al ejecutar, Google te va a pedir permiso: elige tu cuenta y acepta. Es el acceso
normal de Colab a tu Drive; nada sale de tu sesión.

In [ ]:
#@title Paso 2: conectar Drive (ejecuta y acepta el permiso)
from google.colab import drive
drive.mount("/content/drive")
print("\nDrive conectado.")

---

## Paso 3 — Decir en qué carpeta de Drive están

Si los pusiste en `Mi unidad/QQP`, déjalo como está. Si usaste otro nombre,
cámbialo (respeta mayúsculas y acentos).

In [ ]:
#@title Paso 3: ubicar la carpeta
CARPETA_EN_DRIVE = "QQP"  #@param {type:"string"}

import os, glob
CARPETA = os.path.join("/content/drive/MyDrive", CARPETA_EN_DRIVE)

if not os.path.isdir(CARPETA):
    print(f"No existe la carpeta: {CARPETA}\n")
    raiz = "/content/drive/MyDrive"
    if os.path.isdir(raiz):
        print("Carpetas que tienes en Mi unidad:")
        for n in sorted(os.listdir(raiz)):
            if os.path.isdir(os.path.join(raiz, n)):
                print("  ", n)
else:
    comprimidos = sorted(glob.glob(os.path.join(CARPETA, "*.rar")) +
                         glob.glob(os.path.join(CARPETA, "*.zip")))
    sueltos = sorted(glob.glob(os.path.join(CARPETA, "*.csv")))
    print(f"Carpeta: {CARPETA}\n")
    if comprimidos:
        print("Comprimidos (hay que extraerlos en el Paso 4):")
        for f in comprimidos:
            print(f"   {os.path.basename(f):<24} {os.path.getsize(f)/1e6:>8,.0f} MB")
    if sueltos:
        print("\nCSV ya listos:")
        for f in sueltos:
            print(f"   {os.path.basename(f):<24} {os.path.getsize(f)/1e6:>8,.0f} MB")
    if not comprimidos and not sueltos:
        print("La carpeta esta vacia o no tiene .rar ni .csv")

---

## Paso 4 — Extraer los comprimidos

Escribe los nombres de los `.rar` que quieres usar, **separados por coma**.
Cada uno trae un año completo.

Si ya subiste los CSV sueltos, deja esto vacío `""` y ejecuta: los copia y ya.

> Extraer un año tarda unos minutos. Los archivos se guardan en la sesión de
> Colab, no en tu Drive, así que no te consume espacio.

In [ ]:
#@title Paso 4: extraer
COMPRIMIDOS = "QQP_2024.rar, QQP_2025.rar"  #@param {type:"string"}

import os, re, glob, shutil, subprocess
from collections import Counter

for nombre in [n.strip() for n in COMPRIMIDOS.split(",") if n.strip()]:
    origen = os.path.join(CARPETA, nombre)
    if not os.path.exists(origen):
        print(f"NO ENCONTRADO: {nombre}")
        continue
    print(f"Extrayendo {nombre} ... (puede tardar varios minutos)")
    r = subprocess.run(["unar", "-q", "-f", "-D", "-o", "datos", origen],
                       capture_output=True, text=True)
    print("  ok" if r.returncode == 0 else
          f"  fallo: {(r.stderr or r.stdout)[:300]}")

# Los comprimidos traen los CSV dentro de subcarpetas (una por anio). Aqui se
# suben todos al nivel de datos/ para que los patrones con comodin los vean.
movidos = 0
for raiz, _, archivos in os.walk("datos", topdown=False):
    if os.path.abspath(raiz) == os.path.abspath("datos"):
        continue
    for a in archivos:
        if a.lower().endswith(".csv"):
            destino = os.path.join("datos", a)
            if not os.path.exists(destino):
                shutil.move(os.path.join(raiz, a), destino)
                movidos += 1
    try:
        os.rmdir(raiz)
    except OSError:
        pass
if movidos:
    print(f"\n{movidos} archivos sacados de sus subcarpetas")

# copiar tambien los CSV sueltos que haya en Drive
for f in glob.glob(os.path.join(CARPETA, "*.csv")):
    destino = os.path.join("datos", os.path.basename(f))
    if not os.path.exists(destino):
        shutil.copy(f, destino)

csvs = sorted(f for f in os.listdir("datos") if f.lower().endswith(".csv"))
print(f"\n{len(csvs)} archivos CSV listos.")

# resumen de los periodos disponibles, para saber que escribir en el Paso 5
periodos = Counter()
for f in csvs:
    m = re.match(r"(\d{2})-(\d{4})", f)
    if m:
        periodos[f"{m.group(1)}-{m.group(2)}"] += 1

if periodos:
    print("\nPeriodos disponibles (mes-anio, y cuantas piezas tiene cada uno):")
    for per in sorted(periodos, key=lambda x: (x[3:], x[:2])):
        print(f"   {per}_*.csv     {periodos[per]} piezas")
    print("\nCopia dos de estos patrones al Paso 5.")
elif csvs:
    print("\nArchivos encontrados:")
    for f in csvs[:20]:
        print("  ", f)
else:
    print("No quedo ningun CSV. Revisa que los comprimidos sean los correctos.")

---

## Paso 5 — Elegir los dos periodos

Los archivos se llaman `MM-AAAA_pieza.csv`, y **cada mes viene partido en 2
piezas**:

```
01-2024_01.csv     enero 2024, parte 1
01-2024_02.csv     enero 2024, parte 2
```

Aquí no escribes un nombre suelto sino un **patrón**: el `*` significa
"cualquier cosa", así que `01-2024_*.csv` agarra las dos piezas.

| Para analizar | Escribes |
|---|---|
| enero 2024 | `01-2024_*.csv` |
| agosto 2025 | `08-2025_*.csv` |

Usa el **mismo mes** en ambos años, para no confundir inflación con temporada.

In [ ]:
#@title Paso 5: los dos periodos
PATRON_BASE   = "01-2024_*.csv"  #@param {type:"string"}
PATRON_ACTUAL = "01-2025_*.csv"  #@param {type:"string"}

import glob, os
if "CARPETA" not in dir():
    raise SystemExit("Falta ejecutar el Paso 3 antes que este.")
ok = True
for etiqueta, patron in (("BASE  (año viejo)", PATRON_BASE),
                         ("ACTUAL(año nuevo)", PATRON_ACTUAL)):
    hallados = sorted(glob.glob(os.path.join("datos", patron)))
    print(f"{etiqueta}:")
    if hallados:
        for f in hallados:
            print(f"   {os.path.basename(f):<24} {os.path.getsize(f)/1e6:>8,.0f} MB")
    else:
        ok = False
        print("   *** ningun archivo coincide ***")
    print()

print("Listo, pasa al Paso 6." if ok else
      "Revisa el patron contra los nombres que salieron en el Paso 4.")

---

## Paso 6 — Elegir qué analizar

**Importante:** un mes de QQP trae millones de precios. Si no filtras nada, se
acaba la memoria. **Pon siempre al menos un filtro.**

Varios términos van separados por espacios: `"DESODORANTE SHAMPOO"` trae los dos.

In [ ]:
#@title Paso 6: filtros
PRODUCTO  = "DESODORANTE"  #@param {type:"string"}
MARCA     = ""             #@param {type:"string"}
CATEGORIA = ""             #@param {type:"string"}
ESTADO    = ""             #@param {type:"string"}
CADENA    = ""             #@param {type:"string"}

POR_CADENA = True      #@param {type:"boolean"}
MIN_OBSERVACIONES = 3  #@param {type:"integer"}

if not any((PRODUCTO, MARCA, CATEGORIA, ESTADO, CADENA)):
    print("AVISO: sin filtros se puede acabar la memoria.")
    print("       Escribe al menos un producto o una categoria.")
else:
    print("Filtros listos. Pasa al Paso 7.")

---

## Paso 7 — Calcular

Verás un contador de filas mientras avanza. Con archivos grandes tarda varios
minutos.

In [ ]:
#@title Paso 7: calcular
import sys, os, subprocess
faltan = [n for n in ("PATRON_BASE", "PATRON_ACTUAL") if n not in dir()]
if faltan:
    raise SystemExit("Falta ejecutar el Paso 5 antes que este.")
if "PRODUCTO" not in dir():
    raise SystemExit("Falta ejecutar el Paso 6 antes que este.")

cmd = [sys.executable, "inflacion_por_marca.py",
       "--base",   os.path.join("datos", PATRON_BASE),
       "--actual", os.path.join("datos", PATRON_ACTUAL),
       "--min-obs", str(MIN_OBSERVACIONES),
       "--csv", "resultado.csv"]

for bandera, valor in (("--producto", PRODUCTO), ("--marca", MARCA),
                       ("--categoria", CATEGORIA), ("--estado", ESTADO),
                       ("--cadena", CADENA)):
    if valor.strip():
        cmd += [bandera] + valor.split()
if POR_CADENA:
    cmd.append("--por-cadena")

proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print("--- detalle del error ---")
    print(proc.stderr[-3000:])

---

## Paso 8 — Guardar el resultado

Lo guarda en tu Drive (para que no se pierda) y te lo descarga.

In [ ]:
#@title Paso 8: guardar y descargar
import os, shutil
if "CARPETA" not in dir():
    raise SystemExit("Falta ejecutar el Paso 3 antes que este.")

if not os.path.exists("resultado.csv"):
    print("Todavia no hay resultado. Ejecuta el Paso 7 primero.")
else:
    destino = os.path.join(CARPETA, "resultado.csv")
    shutil.copy("resultado.csv", destino)
    print(f"Guardado en tu Drive: {destino}")
    from google.colab import files
    files.download("resultado.csv")

---

## Cómo leer la tabla de reduflación

| Columna | Qué significa |
|---|---|
| **ETIQUETA** | Cuánto subió el precio que ves en el anaquel |
| **CONTENIDO** | Cuánto cambió el tamaño del empaque (negativo = encogió) |
| **REAL** | Cuánto subió el precio por gramo o mililitro |
| **BRECHA** | REAL menos ETIQUETA — la inflación que no se ve |

Una marca con `<-- ENCOGIO` redujo el contenido. Si además su BRECHA es grande,
estás pagando bastante más por gramo aunque el precio del anaquel casi no se
haya movido.

## Notas

- Se usa la **mediana** de precios, no el promedio, para que unos pocos registros
  mal capturados no distorsionen el resultado.
- Un artículo solo aparece si está en **ambos** periodos con al menos
  `MIN_OBSERVACIONES` registros.
- Los archivos extraídos viven solo mientras dure la sesión de Colab. Tu Drive
  conserva los `.rar` originales y el `resultado.csv`.

Código y documentación: <https://github.com/carsam68-MonheyB/CALCULA_TU_INFLACI-N>